In [4]:
import os
import pandas as pd
import re

def filter_real_genes(df, gene_col="feature_name"):
    pattern = r"^BLANK_|^NegControlCodeword_|^NegControlProbe_"

    s = df[gene_col]

    # handle bytes -> str
    if pd.api.types.is_object_dtype(s):
        s = s.map(lambda v: v.decode("utf-8") if isinstance(v, (bytes, bytearray)) else v)

    s = s.astype("string")

    keep = ~s.str.contains(pattern, regex=True, na=False)
    out = df.loc[keep].copy()
    out[gene_col] = s.loc[keep].astype(str).values

    return out


def load_filter_and_save_transcripts(
    parquet_path,
    out_dir,
    gene_col="feature_name",
    skip_if_exists=True,
):
    """
    Load a transcripts.parquet, filter out non-gene entries,
    and save into out_dir/<run_id>/<sample_id>/<kind>/.

    If the cleaned file already exists, skip processing.
    """
    # infer context
    parts = os.path.normpath(parquet_path).split(os.sep)

    # find Xenium sample folder
    idx = next(
        i for i in range(len(parts) - 1, -1, -1)
        if parts[i].startswith("output-")
    )

    run_id = parts[idx - 1]
    sample_id = parts[idx]
    kind = parts[idx + 1]

    out_subdir = os.path.join(out_dir, run_id, sample_id, kind)
    os.makedirs(out_subdir, exist_ok=True)

    out_path = os.path.join(out_subdir, "transcripts_genes_only.parquet")

    # ✅ NEW: skip if already generated
    if skip_if_exists and os.path.exists(out_path):
        print(f"⏭️  Exists → {run_id}/{sample_id}/{kind}")
        return out_path

    print(f"📥 Processing: {run_id}/{sample_id}/{kind}")

    df = pd.read_parquet(parquet_path)
    df_filt = filter_real_genes(df, gene_col=gene_col)
    df_filt.to_parquet(out_path)

    print(f"💾 Saved: {out_path}")
    print(f"   Kept: {len(df_filt):,} / {len(df):,}")

    return out_path

In [5]:
import os

base_dir = "/Volumes/Castelo_Branco/NGSDATA/[spatialOmics_CML]MS_spinal_cord_Xenium_human_brain_panel"
out_dir  = "/Volumes/processing2/hm-xenium-ms/transcripts_cleaned"

saved_files = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file == "transcripts.parquet":
            parquet_path = os.path.join(root, file)
            out_path = load_filter_and_save_transcripts(parquet_path, out_dir=out_dir, gene_col="feature_name")
            saved_files.append(out_path)

📥 Processing: 20230824__123243__20230824_GONCALO_PETRA_run3/output-XETG00047__0010872__Ctrl-C_2017-16__20230824__123323/transcripts.parquet
💾 Saved: /Volumes/processing2/hm-xenium-ms/transcripts_cleaned/20230824__123243__20230824_GONCALO_PETRA_run3/output-XETG00047__0010872__Ctrl-C_2017-16__20230824__123323/transcripts.parquet/transcripts_genes_only.parquet
   Kept: 23,164,794 / 23,772,173
📥 Processing: 20230824__123243__20230824_GONCALO_PETRA_run3/output-XETG00047__0010869__Ctrl-C_2012-071__20230824__123323/transcripts.parquet
💾 Saved: /Volumes/processing2/hm-xenium-ms/transcripts_cleaned/20230824__123243__20230824_GONCALO_PETRA_run3/output-XETG00047__0010869__Ctrl-C_2012-071__20230824__123323/transcripts.parquet/transcripts_genes_only.parquet
   Kept: 28,508,694 / 29,878,259
📥 Processing: 20230829__105411__20230829_Goncalo_Petra_run2/output-XETG00047__0010759__MS-C_2012-078__20230829__105447/transcripts.parquet
💾 Saved: /Volumes/processing2/hm-xenium-ms/transcripts_cleaned/20230829__